In [1]:
import litellm
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    litellm.openai_key = os.getenv("OPENAI_API_KEY")

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [2]:
from datasets import load_dataset

dataset = load_dataset('sms_spam', split=['train'])[0]

for entry in dataset.select(range(3)):
    sms = entry['sms']
    label = entry['label']
    print(f'label={label}, sms={sms}')

Generating train split: 100%|██████████| 5574/5574 [00:00<00:00, 340167.77 examples/s]

label=0, sms=Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

label=0, sms=Ok lar... Joking wif u oni...

label=1, sms=Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's



In [3]:
id2label = {
    0: 'NOT SPAM', 1: 'SPAM',
}
label2id = {
    'NOT SPAM': 0, 'SPAM': 1,
}

for entry in dataset.select(range(3)):
    sms = entry['sms']
    label_id = entry['label']
    print(f'label-{id2label[label_id]}, sms={sms}')

label-NOT SPAM, sms=Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

label-NOT SPAM, sms=Ok lar... Joking wif u oni...

label-SPAM, sms=Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's



In [4]:
def get_sms_messages_string(dataset, item_numbers):
    sms_messages_string = ''
    for item_number, entry in zip(item_numbers,dataset.select(item_numbers)):
        sms = entry['sms']
        label_id = entry['label']

        sms_messages_string += f'{item_number} -> {sms}\n'

    return sms_messages_string

print(get_sms_messages_string(dataset, [0,1,2]))

0 -> Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

1 -> Ok lar... Joking wif u oni...

2 -> Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's




In [6]:
sms_messages_string = get_sms_messages_string(dataset, range(7, 15))


SYSTEM_PROMPT = """You are a helpful assistant that classifies messages as SPAM or NOT SPAM.
Respond only with a JSON object where the keys are the message numbers and the values are the classification."""

USER_PROMPT = sms_messages_string

print("SYSTEM PROMPT:")
print(SYSTEM_PROMPT)
print("USER PROMPT:")
print(USER_PROMPT)

SYSTEM PROMPT:
You are a helpful assistant that classifies messages as SPAM or NOT SPAM.
Respond only with a JSON object where the keys are the message numbers and the values are the classification.
USER PROMPT:
7 -> As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune

8 -> WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.

9 -> Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030

10 -> I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've cried enough today.

11 -> SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info

12 -> URGENT! You have won a 1 week

In [7]:
from litellm import completion

response = completion(
    model=OPENAI_MODEL,
    messages=[
        {
            'role': 'system', 'content': SYSTEM_PROMPT,
        },
        {
            'role': 'user', 'content': USER_PROMPT,
        }
    ],
)

response = response.choices[0].message.content

print(response)

try:
    import json

    json.loads(response)
    print('Response is valid JSON')
except json.JSONDecodeError:
    print('Response is not valid JSON')


{
  "7": "SPAM",
  "8": "SPAM",
  "9": "SPAM",
  "10": "NOT SPAM",
  "11": "SPAM",
  "12": "SPAM",
  "13": "NOT SPAM",
  "14": "NOT SPAM"
}
Response is valid JSON


In [8]:
def get_accuracy(response, dataset, original_indices):
    correct = 0
    total = 0

    if isinstance(response, str):
        import json

        try:
            response = json.loads(response)
        except json.JSONDecodeError as e:
            print("Error decoding JSON response:", e)
            return

    for entry_number, prediction in response.items():
        if int(entry_number) not in original_indices:
            continue

        label_id = dataset[int(entry_number)]["label"]
        label = id2label[label_id]

        # If the prediction from the LLM matches the label in the dataset
        # we increment the number of correct predictions.
        # (Since LLMs do not always produce the same output, we use the
        # lower case version of the strings for comparison)
        if prediction.lower() == label.lower():
            correct += 1
        else:
            print(
                f"Mismatch for entry {entry_number}: predicted={prediction}, actual={label}"
            )

        # increment the total number of predictions
        total += 1

    try:
        accuracy = correct / total
    except ZeroDivisionError:
        print("No matching results found!")
        return

    return round(accuracy, 2)


print(f"Accuracy: {get_accuracy(response, dataset, range(7, 15))}")

Mismatch for entry 7: predicted=SPAM, actual=NOT SPAM
Accuracy: 0.88
